# Python and CLI for Data Science - Session 05

- *Course*: Python and CLI for Data Science
- *Session*: 05
- *Unit*: NumPy Broadcasting

### (Re)sources:
- [Official User Guide](https://numpy.org/doc/stable/user/index.html)
- [Official Beginner's Guide](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/index.html) _by Jake VanderPlas (Code released under the MIT License)_


# Computation with NumPy Arrays: Broadcasting

- *Broadcasting* is a set of rules by which NumPy lets you apply binary operations between arrays of different sizes and shapes

In [ ]:
import numpy as np

## Motivation

- Let's say we have the global average temperatures per day for the last 10 years

In [ ]:
temperature = np.random.normal(14, 3, (10, 12, 31))
temperature.shape

- We can aggregate over dimensions to, for example, get the average temperature per month using np.mean

In [ ]:
average_per_month = np.mean(temperature, axis=2)
average_per_month.shape

- Say we wanted to have the temperature difference of every day from the mean temperature of that month

- Subtracting the two from another does not work, because the shapes do not match

In [ ]:
temperature - average_per_month

- We could loop over the array, but this is very slow (for larger datasets) and inelegant

In [ ]:
temperature_offset_per_day = np.empty_like(temperature)
for year in range(10):
    for month in range(12):
        for day in range(31):
            temperature_offset_per_day[year, month, day] = (
                temperature[year, month, day] - average_per_month[year, month]
            )
print(
    f"Maximum offset: {np.max(temperature_offset_per_day)}",
    f"\nMinimum offset: {np.min(temperature_offset_per_day)}",
)

- How can we elegantly combine two differently shaped arrays?

## Introducing Broadcasting

- Recall that for arrays of the same size, binary operations are performed on an element-by-element basis

In [ ]:
a = np.array([0, 1, 2])
b = np.array([5, 5, 5])
a + b

- Broadcasting allows these types of binary operations to be performed on arrays of different sizes
- For example, we can just as easily add a scalar (think of it as a zero-dimensional array) to an array

In [ ]:
a + 5

- We can think of this as an operation that stretches or duplicates the value `5` into the array `[5, 5, 5]`
- We can similarly extend this idea to arrays of higher dimension
- Observe the result when we add a one-dimensional array to a two-dimensional array

In [ ]:
M = np.ones((3, 3))
M, a

In [ ]:
M + a

- Here the one-dimensional array `a` is stretched, or broadcasted, across the second dimension in order to match the shape of `M`

- While these examples are relatively easy to understand, more complicated cases can involve broadcasting of both arrays

In [ ]:
a = np.arange(3).reshape(3, 1)
b = np.arange(3)
a, b

In [ ]:
a + b

- Here we've broadcasted *both* `a` and `b` to match a common shape, and the result is a two-dimensional array!

- The geometry of these examples is visualized in the following figure

![Broadcasting Visual](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/figures/02.05-broadcasting.png?raw=1)

**Note** 

- NumPy broadcasting does not actually copy the broadcasted values in memory making broadcasting very efficient

## Rules of Broadcasting

- Broadcasting in NumPy follows a strict set of rules to determine the interaction between the two arrays

1. If the two arrays differ in their number of dimensions, the shape of the one with fewer dimensions is *padded* with ones on its leading (left) side

2. If the shape of the two arrays does not match in any dimension, the array with shape equal to 1 in that dimension is stretched to match the other shape

3. If in any dimension the sizes disagree and neither is equal to 1, an error is raised

- To make these rules clear, let's consider a few examples in detail

### Broadcasting Example 1

- Let's consider an operation on two arrays, which have the following shapes
    - `M.shape` is `(2, 3)`
    - `a.shape` is `(3,)`

- We see by rule 1 that the array `a` has fewer dimensions, so we pad it on the left with ones
    - `M.shape` remains `(2, 3)`
    - `a.shape` becomes `(1, 3)`

- By rule 2, we now see that the first dimension disagrees, so we stretch this dimension to match
    - `M.shape` remains `(2, 3)`
    - `a.shape` becomes `(2, 3)`

- The shapes now match, and we see that the final shape will be `(2, 3)`

In [ ]:
M = np.ones((2, 3))
a = np.arange(3)
M + a

### Broadcasting Example 2

- Now let's take a look at an example where both arrays need to be broadcast
    - `a.shape` is `(3, 1)`
    - `b.shape` is `(3,)`

- Rule 1 says we must pad the shape of `b` with ones
    - `a.shape` remains `(3, 1)`
    - `b.shape` becomes `(1, 3)`

- And rule 2 tells us that we must update each of these ``1``s to match the corresponding size of the other array
    - `a.shape` becomes `(3, 3)`
    - `b.shape` becomes `(3, 3)`

- Because the results match, these shapes are compatible and we get an array of shape `(3, 3)`

In [ ]:
a = np.arange(3)[:, np.newaxis]
b = np.arange(3)
a + b

### Broadcasting Example 3

- Next, let's take a look at an example in which the two arrays are not compatible
- This is just a slightly different situation than in the first example: the matrix `M` is transposed
    - `M.shape` is `(3, 2)`
    - `a.shape` is `(3,)`

- Again, rule 1 tells us that we must pad the shape of `a` with ones
    - `M.shape` remains `(3, 2)`
    - `a.shape` becomes `(1, 3)`

- By rule 2, the first dimension of `a` is then stretched to match that of `M`
    - `M.shape` remains `(3, 2)`
    - `a.shape` becomes `(3, 3)`

- Now we hit rule 3—the final shapes do not match, so these two arrays are incompatible

In [ ]:
M = np.ones((3, 2))
a = np.arange(3)
M + a

- You could imagine making `a` and `M` compatible by padding `a`'s shape with ones on the right rather than the left

- But this is not how the broadcasting rules work!
- That sort of flexibility would lead to potential areas of ambiguity

- Right-side padding is can be done explicitly by reshaping the array (we'll use the `np.newaxis` keyword for this)

In [ ]:
a[:, np.newaxis].shape # same as a.reshape(3, 1)

In [ ]:
M + a[:, np.newaxis]

- Also note that we've been focusing on the `+` operator here
- Broadcasting rules apply to *all* binary ufuncs

- For example, here is the `logaddexp(a, b)` function, which computes `log(exp(a) + exp(b))` with more precision than the naive approach

In [ ]:
np.logaddexp(M, a[:, np.newaxis])

### Centering an Array

- One common example in data science is subtracting the row-wise mean from an array of data
- Imagine we have an array of 10 observations, each of which consists of 3 values
- We'll store this in a $10 \times 3$ array

In [ ]:
X = np.random.random((10, 3))
X

- We can compute the mean of each column using the `mean` aggregate across the first dimension

In [ ]:
Xmean = X.mean(axis=0)
Xmean

- And now we can center the `X` array by subtracting the mean (this is a broadcasting operation)

In [ ]:
X_centered = X - Xmean
X_centered

- To double-check that we've done this correctly, we can check that the centered array has a mean near zero

In [ ]:
X_centered.mean(0)

### Temperature Offset Example

- We can now also complete our example from our motivation section
- We can compute the temperature difference per day from the average temperature (per year or month) by broadcasting the average temperatures over the days dimension

In [ ]:
temperature.shape

In [ ]:
# keepsdims=True keeps 1 at collapsed dimension
average_per_month = np.mean(temperature, axis=2, keepdims=True)
print(f"Average shape: {average_per_month.shape}")
temperature_offset = temperature - average_per_month
print(f"Offset shape: {temperature_offset.shape}")
print(
    f"Maximum offset: {np.max(temperature_offset_per_day)}",
    f"\nMinimum offset: {np.min(temperature_offset_per_day)}",
)

### Exercises

1. Scale the rows of the following matrix to obtain the other matrix
```
[1 2 3]             [ 2  4  6]
[4 5 6]  -scale->   [12 15 18]
[7 8 9]             [28 32 36]
```

In [ ]:
# Solve the first exercise here
M = np.arange(1, 10).reshape(3, 3)

In [ ]:
M * np.arange(2, 5)[:, np.newaxis]

2. Compute the pairwise distances between all points of $5 \times 2$ array of random Cartesian coordinates. The result should be a $5 \times 5$ matrix where the entry at $i,j$ contains the distance of i-th coordinate to the j-th coordinate

For example, for the two points (1, 2) and (2, 2), the result should be
```
[[0 1]
 [1 0]]
```
since the diagonal represents the distance from a point to itself and the euclidean distance between (1, 2) and (2, 2) is 1.

In [ ]:
# Solve the second exercise here
C = np.random.random((5, 2))
np.sqrt(((C.reshape(5, 1, 2) - C.reshape(1, 5, 2)) ** 2).sum(axis=2))

In [ ]:
np.sqrt(((C[:, np.newaxis] - C[np.newaxis]) ** 2).sum(axis=2))